# 2.2 · 分布实战：拟合与选择 / Distributions in Practice

> **课程定位**
> 0.9 节"认识"了 10 大分布；本课解决实战问题：**拿到一列真实数据，怎么判断它服从什么分布、怎么拟合参数、怎么检验拟合好坏**。这是定价模型、风控、排队论、可靠性分析的日常。
> 0.9 introduced the distributions; this lesson answers the practical question: given real data, which distribution, what parameters, and how good is the fit?

> 💡 **面试相关**
> - "怎么判断数据是不是正态" ★★★★（QQ 图 + 检验）
> - "长尾数据怎么建模" ★★★★（log 变换 / lognormal）
> - "scipy 的 frozen distribution 是什么" ★★★

---

## 目录
1. [scipy.stats 统一接口 ⭐](#1)
2. [分布关系地图](#2)
3. [QQ 图：判断分布的首选工具 ⭐](#3)
4. [正态性检验：Shapiro / KS / Anderson](#4)
5. [参数拟合：`fit()` 与 MLE](#5)
6. [模型选择：AIC + KS 距离](#6)
7. [实战：给三种真实形态的数据挑分布](#7)
8. [小结](#8)


<a id="1"></a>
## 1. scipy.stats 统一接口 ⭐ / The Unified Interface

scipy 里**每个分布都有同一套方法**——学一遍走天下：

| 方法 | 含义 | 数学 |
|---|---|---|
| `.pdf(x)` / `.pmf(k)` | 密度 / 质量 | $p(x)$ |
| `.cdf(x)` | 累积分布 | $\Pr(X \le x)$ |
| `.ppf(q)` ⭐ | 分位数（CDF 的反函数）| $\inf\{x: F(x) \ge q\}$ |
| `.rvs(size)` | 采样 | — |
| `.fit(data)` | MLE 拟合参数 | $\arg\max \mathcal{L}$ |
| `.mean() / .std() / .stats()` | 理论矩 | — |

**frozen distribution**：把参数"冻"进对象——`st.norm(loc=100, scale=15)` 返回一个固定参数的分布对象，之后所有方法不用再传参。
A frozen distribution locks the parameters in, so later calls skip them.


In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

iq = st.norm(loc=100, scale=15)          # frozen: IQ 分布

print(f"P(IQ ≤ 130)        = {iq.cdf(130):.4f}")
print(f"P(IQ > 145)        = {iq.sf(145):.6f}      # sf = 1-cdf, 数值上更准")
print(f"前 1% 的门槛        = {iq.ppf(0.99):.1f}")
print(f"中间 95% 区间       = [{iq.ppf(0.025):.1f}, {iq.ppf(0.975):.1f}]")
print(f"理论 mean/std       = {iq.mean()}, {iq.std()}")


> 💡 **`.sf()`（survival function）比 `1 - cdf` 数值更稳**：尾部概率小到 1e-15 时，`1 - cdf` 会因浮点抵消变成 0，`sf` 直接算尾部不丢精度。风控/极值场景必用。
> `.sf()` beats `1 - cdf` numerically: tiny tail probabilities survive instead of cancelling to zero.

> 💡 **`.ppf()` 是被低估的主力**：算 VaR（"99% 置信下最多亏多少"）、检验临界值（2.6 节）、逆变换采样（2.12 节）全靠它。


<a id="2"></a>
## 2. 分布关系地图 / The Distribution Relationship Map

分布不是孤岛——它们由**生成机制**串联。认机制比背公式有用：
Distributions aren't islands — they're linked by generative mechanisms:

```
Bernoulli(p) ──n 次求和──→ Binomial(n,p) ──n→∞, np=λ──→ Poisson(λ)
                              │                            │
                              └──n→∞ (CLT)──→ Normal ←──事件间隔──┘
                                               │         Exponential(λ)
   "乘性"过程 (增长率相乘)                       │              │
   X = ∏ factors → log X = Σ log ──CLT──→ LogNormal      k 个间隔求和
                                               │              ↓
                                          X², 求和 → Chi² ← Gamma(k, λ)
```

| 机制 / Mechanism | 产生 / Yields | 真实例子 |
|---|---|---|
| 大量独立因素**相加** | Normal | 测量误差、身高 |
| 大量独立因素**相乘** | LogNormal ⭐ | 收入、城市规模、文件大小、股价 |
| 单位时间稀有事件计数 | Poisson | 客服来电、网站错误数 |
| 泊松事件的**等待时间** | Exponential | 设备故障间隔、到店间隔 |
| 第 k 次事件的等待 | Gamma | 修复总时长 |
| "最大值"的分布 | Gumbel/Fréchet（极值）| 洪水、最大回撤 |

> 💡 面试题"用户消费金额怎么建模"的好答案：**乘性机制 → 先 log 再看**——log 后接近正态就是 lognormal。


<a id="3"></a>
## 3. QQ 图：判断分布的首选工具 ⭐ / QQ Plots

**原理**：把**样本分位数**对**理论分位数**画散点。若数据真来自该分布 → 点落在直线上。

$$\text{plot}\Big(F^{-1}\big(\tfrac{i - 0.5}{n}\big),\; x_{(i)}\Big), \quad i = 1, \dots, n$$

**读图口诀**（对照正态 QQ 图）：
- 两端**上翘+下垂**（S 反向）→ **重尾**
- 整体**弯成弧**→ **偏态**（凹向上 = 右偏）
- 中间直、两端歪 → 中心近似正态但尾部异常 ⭐ 最常见


In [ ]:
# 四种数据的正态 QQ 图 / Normal QQ plots for four data shapes
datasets = {
    "Normal (对照)":   rng.normal(0, 1, 500),
    "Student-t df=3":  rng.standard_t(3, 500),
    "LogNormal":       rng.lognormal(0, 0.7, 500),
    "Uniform":         rng.uniform(-2, 2, 500),
}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, (name, d) in zip(axes, datasets.items()):
    st.probplot(d, dist="norm", plot=ax)
    ax.set_title(name, fontsize=10); ax.get_lines()[0].set(markersize=3, alpha=0.6)
plt.suptitle("Normal QQ plots — deviations tell the story", y=1.04)
plt.tight_layout(); plt.show()


**逐图解读**：
- **t(3)**：两端剧烈偏离直线（左端更低、右端更高）= 重尾签名
- **LogNormal**：整体上凹弧线 = 右偏签名
- **Uniform**：两端"卷回来"（S 形）= 轻尾（没有尾巴）

> 💡 QQ 图 > 直方图的原因：**尾部行为在直方图上看不见**（bin 里就几个点），QQ 图把尾部拉到对角线两端放大展示。


<a id="4"></a>
## 4. 正态性检验 / Normality Tests

| 检验 | 原理 | 特点 |
|---|---|---|
| **Shapiro-Wilk** ⭐ | 顺序统计量与正态分位数的相关 | 小中样本最有功效；$n>5000$ scipy 拒绝跑 |
| **KS** (Kolmogorov-Smirnov) | $\max_x \lvert F_n(x) - F(x) \rvert$ | 通用但保守；⚠ 参数若从数据估计 p 值不准 |
| **Anderson-Darling** | 加权 KS，**加重尾部** | 对尾部偏离敏感 |

⚠ **大样本悖论**：$n$ 很大时，**任何**真实数据都会被拒绝正态（真实世界没有完美正态）——p 值小不等于偏离重要。**先看 QQ 图定性，检验只做辅助**。
With huge $n$, every real dataset fails normality tests. Eyeball the QQ plot first; tests are secondary.


In [ ]:
# 三个检验对四种数据 / Three tests on four datasets
print(f"{'data':<16} {'Shapiro p':>10} {'KS p':>10} {'Anderson stat':>14}  (A-D 5% 临界值见下)")
print("-" * 60)
for name, d in datasets.items():
    sh_p = st.shapiro(d).pvalue
    # KS: 参数从数据估计时要标准化后对 N(0,1) 检验（近似）
    z = (d - d.mean()) / d.std(ddof=1)
    ks_p = st.kstest(z, "norm").pvalue
    ad = st.anderson(d, dist="norm")
    print(f"{name:<18} {sh_p:>10.2e} {ks_p:>10.2e} {ad.statistic:>12.2f}")

ad0 = st.anderson(datasets["Normal (对照)"], dist="norm")
print(f"\nAnderson 5% 临界值 = {ad0.critical_values[2]:.3f} (statistic 超过即拒绝正态)")


<a id="5"></a>
## 5. 参数拟合：`fit()` 与 MLE / Parameter Fitting

`dist.fit(data)` 在背后做**最大似然估计**（2.9 节推导数学）。

⚠ **`fit` 的最大坑**：scipy 连续分布都带 `loc`（平移）和 `scale`（缩放）参数，**默认全部自由拟合**。很多分布（lognormal、gamma、expon）物理上 `loc` 应该是 0——**不固定 `floc=0` 会拟出荒谬参数**。
Almost every scipy continuous distribution has free loc/scale; for lognormal/gamma/expon you usually must pin `floc=0` or get nonsense.


In [ ]:
# 演示 floc 的重要性 / Why floc=0 matters
true_data = rng.lognormal(mean=1.0, sigma=0.5, size=2000)     # 真参数 σ=0.5, scale=e^1≈2.72

bad = st.lognorm.fit(true_data)                # loc 自由 → 经常跑偏
good = st.lognorm.fit(true_data, floc=0)       # 固定 loc=0 ✓

print(f"真参数            : shape(σ)=0.5, loc=0, scale=e^1={np.e:.3f}")
print(f"fit (loc 自由)    : shape={bad[0]:.3f}, loc={bad[1]:.3f}, scale={bad[2]:.3f}")
print(f"fit (floc=0)      : shape={good[0]:.3f}, loc={good[1]:.3f}, scale={good[2]:.3f}   ← 准确恢复")


<a id="6"></a>
## 6. 模型选择：AIC + KS 距离 / Model Selection

候选分布都拟合一遍，谁好？两个互补的标尺：

**AIC**（Akaike Information Criterion）——拟合好坏与复杂度的平衡：
$$\mathrm{AIC} = 2k - 2\ln \hat{\mathcal{L}}, \qquad k = \text{参数个数，越小越好}$$

**KS 距离**——经验 CDF 和拟合 CDF 的最大间隙（越小越好），直观可画。

> 💡 AIC 只能**同一份数据上横向比较**，绝对值无意义。差 2 以内算打平，差 10 以上是压倒性。


In [ ]:
def fit_candidates(data, candidates):
    # 对每个候选分布: MLE 拟合 → AIC + KS 距离
    rows = []
    for name, dist, kwargs in candidates:
        params = dist.fit(data, **kwargs)
        ll = np.sum(dist.logpdf(data, *params))
        aic = 2 * len(params) - 2 * ll
        ks = st.kstest(data, dist.name, args=params).statistic
        rows.append({"dist": name, "AIC": aic, "KS_dist": ks, "params": np.round(params, 3)})
    return pd.DataFrame(rows).sort_values("AIC").reset_index(drop=True)

candidates = [
    ("normal",    st.norm,    {}),
    ("lognormal", st.lognorm, {"floc": 0}),
    ("gamma",     st.gamma,   {"floc": 0}),
    ("expon",     st.expon,   {"floc": 0}),
]
print(fit_candidates(true_data, candidates))


**lognormal AIC 最小 + KS 最小** —— 正确选中真模型，gamma 是"亚军"（gamma 和 lognormal 形状常常接近，真实工作中两者难分是常态）。
Lognormal wins on both criteria; gamma comes close — these two are notoriously hard to tell apart in practice.


<a id="7"></a>
## 7. 实战：给三种真实形态的数据挑分布 / Hands-on

模拟三种典型业务数据形态（按真实机制生成），**假装不知道真相**，走完整流程：QQ 定性 → 候选拟合 → AIC/KS 选择 → 可视化验证。


In [ ]:
# 三种业务数据 / Three business-realistic datasets
biz = {
    # 客服每小时来电数: 计数 → 泊松机制
    "calls/hour":  rng.poisson(lam=6.5, size=1500).astype(float),
    # 用户会话时长: 乘性 → lognormal 机制
    "session_min": rng.lognormal(mean=1.2, sigma=0.8, size=1500),
    # 服务器响应: gamma 机制 (多阶段处理之和)
    "latency_ms":  rng.gamma(shape=3.0, scale=12.0, size=1500),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, (name, d) in zip(axes, biz.items()):
    ax.hist(d, bins=60, density=True, alpha=0.75)
    ax.set_title(f"{name}\nskew={st.skew(d):.2f}")
plt.tight_layout(); plt.show()


In [ ]:
# 连续两列走完整选择流程 / Full selection on the two continuous columns
for name in ["session_min", "latency_ms"]:
    print(f"=== {name} ===")
    print(fit_candidates(biz[name], candidates).head(3).to_string(index=False))
    print()


In [ ]:
# 离散列: 计数数据对比 Poisson 拟合 / Count column: fit Poisson
calls = biz["calls/hour"].astype(int)
lam_hat = calls.mean()                       # Poisson 的 MLE 就是样本均值 / Poisson MLE = sample mean
print(f"λ̂ = {lam_hat:.3f}  (真值 6.5)")
print(f"Poisson 性质检查: mean={calls.mean():.2f} ≈ var={calls.var(ddof=1):.2f} ?  "
      f"(Poisson 的 mean = var；比值 {calls.var(ddof=1)/calls.mean():.2f}, ≈1 ✓)")

# 拟合可视化 / Overlay fit
k = np.arange(0, calls.max()+1)
plt.figure(figsize=(8, 3.5))
plt.hist(calls, bins=np.arange(-0.5, calls.max()+1.5), density=True, alpha=0.7, label="data")
plt.plot(k, st.poisson(lam_hat).pmf(k), "ro-", ms=5, label=f"Poisson(λ̂={lam_hat:.2f})")
plt.legend(); plt.title("calls/hour vs fitted Poisson")
plt.tight_layout(); plt.show()


> 💡 **方差/均值比（dispersion ratio）**是计数数据的快速诊断：≈1 → Poisson；**>1（过散）→ 负二项**（真实业务数据多数过散：用户行为异质性）；<1 → 欠散（少见）。
> The variance/mean ratio is the quick count-data diagnostic: ≈1 Poisson, >1 negative binomial (most real behavioral counts are over-dispersed).


In [ ]:
# 最终验证: 经验 CDF vs 拟合 CDF / Final check: empirical vs fitted CDF
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, name, dist, kwargs in [
    (axes[0], "session_min", st.lognorm, {"floc": 0}),
    (axes[1], "latency_ms",  st.gamma,   {"floc": 0}),
]:
    d = np.sort(biz[name])
    params = dist.fit(d, **kwargs)
    ecdf = np.arange(1, len(d)+1) / len(d)
    ax.plot(d, ecdf, lw=1.5, label="empirical CDF")
    ax.plot(d, dist.cdf(d, *params), "--", lw=1.5, label=f"fitted {dist.name}")
    ax.set_title(name); ax.legend()
plt.tight_layout(); plt.show()
print("两条曲线几乎重合 = 拟合可信 / near-overlap = trustworthy fit")


<a id="8"></a>
## 8. 小结 / Summary

```
拿到一列数据
  1. 画直方图 + QQ 图 → 定性: 对称? 偏? 重尾? 离散?
  2. 想生成机制 → 缩小候选 (加→normal, 乘→lognormal, 计数→poisson, 等待→expon/gamma)
  3. dist.fit(data, floc=0) → MLE 参数     ⚠ 记得固定 loc
  4. AIC + KS 横向比较候选
  5. 经验 CDF vs 拟合 CDF 叠图 → 终检
```

### 💡 面试速查
1. **判断正态**：QQ 图为主（大样本下检验必拒）；Shapiro 小样本最有功效
2. **scipy 五件套**：pdf/cdf/**ppf**/rvs/fit + frozen distribution
3. **`sf()` 算尾概率**不丢精度
4. **长尾数据**：先 log；log 后正态 = lognormal（乘性机制）
5. **计数数据**：var/mean ≈1 Poisson，>1 负二项
6. **AIC 只能同数据横比**，差 10+ 才算压倒

### 下一节
**2.3 LLN & CLT 深挖**——0.9 见过 CLT 动画；这次讲**收敛速度、标准误、CLT 什么时候失效**（Cauchy 的惊悚案例）。
